# A/B-тест: геймификация в EdTech-платформе

Анализ результатов A/B-теста продуктовой гипотезы: добавление элементов геймификации (система достижений за регулярное решение задач) в личный кабинет ученика повышает долю завершённых домашних заданий и удержание пользователей на платформе.

**Данные:** логи событий и распределение пользователей по группам за 3 недели эксперимента, 6008 пользователей до очистки.

**Инструменты:** Python, pandas, SciPy, statsmodels, matplotlib.

**Результат:** статистически и практически значимый рост по всем трём ключевым метрикам в группе с геймификацией. Рекомендация — раскатить фичу на всех пользователей платформы.

## Содержание

1. [Шаг 1. Подготовка данных](#Шаг-1.-Подготовка-данных)
2. [Шаг 2. Sanity check](#Шаг-2.-Sanity-check)
3. [Шаг 3. Метрики на уровне пользователя](#Шаг-3.-Метрики-на-уровне-пользователя)
4. [Шаг 4. Разведочный анализ](#Шаг-4.-Разведочный-анализ)
5. [Шаг 5. Выбор статистического критерия](#Шаг-5.-Выбор-статистического-критерия)
6. [Шаг 6. Статистический расчёт](#Шаг-6.-Статистический-расчёт)
7. [Шаг 7. Интерпретация результатов](#Шаг-7.-Интерпретация-результатов)
8. [Шаг 8. Бизнес-рекомендация](#Шаг-8.-Бизнес-рекомендация)

## Шаг 1. Подготовка данных

Объединяем лог событий со справочником пользователей и очищаем данные: убираем пользователей с конфликтующей группой (встречаются и в A, и в B), полные дубли строк в логе и ботов — аккаунты с аномально высокой активностью (700+ событий за 3 недели при норме до 52).

In [ ]:
import pandas as pd

users = pd.read_csv('/content/drive/MyDrive/apl/ab_test_users_groups.csv')
events = pd.read_csv('/content/drive/MyDrive/apl/ab_test_events.csv', parse_dates=['timestamp'])
users['signup_date'] = pd.to_datetime(users['signup_date'])

conflict_ids = users[users['user_id'].duplicated(keep=False)]['user_id'].unique()

events = events.drop_duplicates()

ev_counts = events.groupby('user_id').size()
bot_ids = ev_counts[ev_counts > 100].index

bad_ids = set(conflict_ids) | set(bot_ids)
users_clean = users[~users['user_id'].isin(bad_ids)]
events_clean = events[~events['user_id'].isin(bad_ids)]

merged = events_clean.merge(users_clean, on='user_id', how='left')

**Итог:** удалено 20 пользователей (8 с конфликтом групп + 12 ботов) и 2 дублирующиеся строки. Осталось 5980 пользователей, баланс групп почти не сдвинулся (A: 2992, B: 2988).

## Шаг 2. Sanity check

Перед тем как доверять результатам эксперимента, проверяем механику сплитования: нет ли пересечения между группами и нет ли перекоса в самом разбиении (Sample Ratio Mismatch, SRM).

In [ ]:
a_ids = set(users_clean[users_clean['group']=='A']['user_id'])
b_ids = set(users_clean[users_clean['group']=='B']['user_id'])
intersection = a_ids & b_ids
print(len(intersection))

**Итог:** пересечений между группами нет.

### SRM-тест

Проверяем гипотезу H0: истинная доля группы A равна 0.5, наблюдаемое отклонение (2992 против 2988) объясняется случайностью. Критерий — хи-квадрат на соответствие ожидаемому распределению, для сверки — точный биномиальный тест.

In [ ]:
from scipy import stats

counts = users_clean['group'].value_counts()
n_a, n_b = counts['A'], counts['B']
total = n_a + n_b

observed = [n_a, n_b]
expected = [total/2, total/2]

chi2, p_value = stats.chisquare(f_obs=observed, f_exp=expected)
print(p_value)

result = stats.binomtest(n_a, total, p=0.5)
print(result.pvalue)

**Итог:** p-value ≈ 0.96 (хи-квадрат) и ≈ 0.97 (биномиальный тест) — отклонений от ожидаемого сплита 50/50 нет, эксперимент настроен корректно.

## Шаг 3. Метрики на уровне пользователя

Агрегируем лог событий в одну строку на пользователя: считаем количество событий каждого типа, добавляем группу и класс, рассчитываем производные метрики — долю решённых домашних заданий (`completion_rate`) и флаги возврата на 7 и 14 день.

In [ ]:
event_counts = events_clean.groupby(['user_id', 'event_type']).size().unstack(fill_value=0)

metrics = event_counts.merge(users_clean, on='user_id', how='left')

metrics['completion_rate'] = metrics['homework_completed'] / metrics['homework_started'].replace(0, pd.NA)

metrics['returned_d7'] = (metrics['return_visit_d7'] > 0).astype(int)
metrics['returned_d14'] = (metrics['return_visit_d14'] > 0).astype(int)

**Итог:** таблица `metrics`, 5980 строк, одна на пользователя, с готовыми продуктовыми метриками для дальнейшего сравнения групп.

## Шаг 4. Разведочный анализ

Смотрим на данные визуально и через описательную статистику до формальных тестов: строим воронку конверсии и распределения ключевых метрик для обеих групп.

In [ ]:
for g in ['A','B']:
    sub = metrics[metrics['group']==g]
    n_total = len(sub)
    n_login = (sub['login']>0).sum()
    n_started = (sub['homework_started']>0).sum()
    n_completed = (sub['homework_completed']>0).sum()
    print(g, n_total, n_login, n_started, n_completed)

**Итог:** login и homework_started — 100% в обеих группах, отсева на этих ступенях нет по построению данных. Разница проявляется на последней ступени, completed ≥1 раз: 97.6% в A против 98.8% в B.

In [ ]:
metrics.groupby('group')['completion_rate'].describe()

**Итог:** среднее completion_rate — 34.6% в группе A и 43.8% в группе B, разница заметна уже на этом уровне.

In [ ]:
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0001]
labels = ['0-20%','20-40%','40-60%','60-80%','80-100%']
metrics['cr_bucket'] = pd.cut(metrics['completion_rate'], bins=bins, labels=labels, include_lowest=True)

for g in ['A','B']:
    sub = metrics[metrics['group']==g]
    print(g)
    print(sub['cr_bucket'].value_counts(normalize=True))

**Итог:** в группе A почти 67% пользователей — в двух нижних корзинах (до 40% решённых задач), в группе B туда попадает 44%. Распределение группы B смещено к более высоким значениям.

In [ ]:
metrics.groupby('group')[['returned_d7','returned_d14']].mean()

**Итог:** Retention D7 — 72.7% (A) против 78.0% (B); Retention D14 — 28.8% (A) против 36.2% (B). Оба показателя выше у группы B.

In [ ]:
import matplotlib.pyplot as plt

table = metrics.groupby('group')['cr_bucket'].value_counts(normalize=True).unstack() * 100

table.T.plot(kind='bar', figsize=(8,5))
plt.xlabel('Completion rate')
plt.ylabel('% пользователей группы')
plt.title('Распределение completion_rate по группам')
plt.xticks(rotation=0)
plt.legend(title='Группа')
plt.show()

**Итог:** график наглядно показывает сдвиг распределения completion_rate вправо у группы B — больше пользователей оказывается в средних и высоких корзинах.

## Шаг 5. Выбор статистического критерия

Перед выбором теста проверяем распределение `completion_rate` на нормальность — от этого зависит, можно ли использовать классический t-тест, или нужен непараметрический аналог.

In [ ]:
from scipy import stats

for g in ['A','B']:
    cr = metrics[metrics['group']==g]['completion_rate']
    shapiro_stat, shapiro_p = stats.shapiro(cr) #это тест Шапиро-Уилка на нормальность распределения ген совокупности
    print(g, shapiro_p)

**Итог:** p-value теста Шапиро-Уилка исчезающе мал в обеих группах — гипотеза о нормальном распределении отвергается. При таком большом размере выборки тест сверхчувствителен даже к небольшим отклонениям, поэтому решение опирается и на этот результат, и на форму распределения с графика на шаге 4.

**Выбор теста:** Mann-Whitney U для `completion_rate` (не требует нормальности), z-тест для двух долей и хи-квадрат для retention (метрики бинарные, вопрос о нормальности к ним не относится).

## Шаг 6. Статистический расчёт

Считаем p-value для каждой метрики выбранными тестами, доверительные интервалы разницы между группами и минимальный обнаружимый эффект (MDE) — какой минимальный по размеру эффект эксперимент вообще был способен поймать при таком размере выборки.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

a = metrics[metrics['group']=='A']
b = metrics[metrics['group']=='B']

# t-тест Уэлча (вариант t-теста, не требующий равенства дисперсий групп — у нас они разные: 0.156 и 0.181)
t_stat, t_p = stats.ttest_ind(a['completion_rate'], b['completion_rate'], equal_var=False)

# Mann-Whitney U
u_stat, u_p = stats.mannwhitneyu(a['completion_rate'], b['completion_rate'], alternative='two-sided')

# z-тест для двух долей
count = [a['returned_d7'].sum(), b['returned_d7'].sum()]
nobs = [len(a), len(b)]
z_stat, z_p = proportions_ztest(count, nobs)

# хи-квадрат на таблице сопряжённости 2х2
table = pd.crosstab(metrics['group'], metrics['returned_d7'])
chi2, chi2_p, dof, expected = stats.chi2_contingency(table, correction=False)

print('=== COMPLETION RATE ===')
print(f'Группа A: среднее={a["completion_rate"].mean():.3f}')
print(f'Группа B: среднее={b["completion_rate"].mean():.3f}')
print(f't-тест: p-value={t_p:.2e}')
print(f'Mann-Whitney U: p-value={u_p:.2e}')
print()

print('=== RETENTION D7 ===')
print(f'Группа A: {a["returned_d7"].mean():.1%}')
print(f'Группа B: {b["returned_d7"].mean():.1%}')
print(f'z-тест: p-value={z_p:.2e}')
print(f'Хи-квадрат: p-value={chi2_p:.2e}')

In [ ]:
# те же тесты для returned_d14
count14 = [a['returned_d14'].sum(), b['returned_d14'].sum()]
z_stat14, z_p14 = proportions_ztest(count14, nobs)

table14 = pd.crosstab(metrics['group'], metrics['returned_d14'])
chi2_14, chi2_p14, dof14, expected14 = stats.chi2_contingency(table14, correction=False)

print('=== RETENTION D14 ===')
print(f'Группа A: {a["returned_d14"].mean():.1%}')
print(f'Группа B: {b["returned_d14"].mean():.1%}')
print(f'z-тест: p-value={z_p14:.2e}')
print(f'Хи-квадрат: p-value={chi2_p14:.2e}')

**Итог:** p-value по всем трём метрикам исчезающе малы (порядка 10⁻⁶–10⁻⁹⁴) — разница между группами статистически значима с большим запасом.

In [ ]:
import numpy as np

mean_a, mean_b = a['completion_rate'].mean(), b['completion_rate'].mean()
var_a, var_b = a['completion_rate'].var(), b['completion_rate'].var()
n_a, n_b = len(a), len(b)

diff = mean_b - mean_a
se = np.sqrt(var_a/n_a + var_b/n_b)
ci_low, ci_high = diff - 1.96*se, diff + 1.96*se

p_a, p_b = a['returned_d7'].mean(), b['returned_d7'].mean()
diff_p = p_b - p_a
se_p = np.sqrt(p_a*(1-p_a)/n_a + p_b*(1-p_b)/n_b)
ci_low_p, ci_high_p = diff_p - 1.96*se_p, diff_p + 1.96*se_p

print('completion_rate:', round(diff,4), 'CI95% = [', round(ci_low,4), ',', round(ci_high,4), ']')
print('returned_d7:', round(diff_p,4), 'CI95% = [', round(ci_low_p,4), ',', round(ci_high_p,4), ']')

In [ ]:
p_a14, p_b14 = a['returned_d14'].mean(), b['returned_d14'].mean()
diff_p14 = p_b14 - p_a14
se_p14 = np.sqrt(p_a14*(1-p_a14)/n_a + p_b14*(1-p_b14)/n_b)
ci_low_p14, ci_high_p14 = diff_p14 - 1.96*se_p14, diff_p14 + 1.96*se_p14

print('returned_d14:', round(diff_p14,4), 'CI95% = [', round(ci_low_p14,4), ',', round(ci_high_p14,4), ']')

**Итог:** ни один из трёх доверительных интервалов не касается нуля — completion_rate [8.3%; 10.1%], Retention D7 [3.1%; 7.5%], Retention D14 [5.1%; 9.8%]. Размер эффекта убедителен сам по себе, поверх формальной значимости.

In [ ]:
z_alpha = stats.norm.ppf(1 - 0.05/2)
z_power = stats.norm.ppf(0.8)

mde_cr = (z_alpha + z_power) * se

In [ ]:
mde_d7 = (z_alpha + z_power) * se_p
mde_d14 = (z_alpha + z_power) * se_p14

print('MDE completion_rate:', round(mde_cr,4))
print('MDE returned_d7:', round(mde_d7,4))
print('MDE returned_d14:', round(mde_d14,4))

**Итог:** наблюдаемая разница по каждой метрике в несколько раз больше MDE — выборка была способна поймать эффект куда мельче того, что реально нашли, запас мощности солидный.

## Шаг 7. Интерпретация результатов

Статистическая значимость отвечает только на вопрос, реальна ли разница. Практическая значимость — отдельный вопрос: достаточно ли велик эффект, чтобы оправдать решение. При большой выборке даже крошечная неважная для бизнеса разница легко становится статистически значимой, поэтому размер эффекта стоит оценивать отдельно, например через относительный прирост.

**Completion rate:** 34.6% → 43.8%, относительный прирост +26.6%. Каждый четвёртый из решивших задачи в группе B, вероятно, не довёл бы их до конца в старом интерфейсе.

**Retention D7:** +5.3 п.п., относительно +7.3%. Реальный, но самый скромный эффект из трёх — базовый уровень уже высокий, места для роста меньше.

**Retention D14:** +7.4 п.п., относительно +25.7%. Самый интересный результат: удержание естественно проседает со временем, геймификация, похоже, замедляет это затухание — разрыв между группами к 14 дню вырос в относительных цифрах, а не сузился. Для бизнеса это особенно ценно: долгосрочное удержание сильнее влияет на пожизненную ценность ученика, чем разовый всплеск активности.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = ['Completion\nrate', 'Retention\nD7', 'Retention\nD14']
values_a = [a['completion_rate'].mean()*100, a['returned_d7'].mean()*100, a['returned_d14'].mean()*100]
values_b = [b['completion_rate'].mean()*100, b['returned_d7'].mean()*100, b['returned_d14'].mean()*100]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8,5))
bars_a = ax.bar(x - width/2, values_a, width, label='Группа A', color='#888780')
bars_b = ax.bar(x + width/2, values_b, width, label='Группа B', color='#1D9E75')

ax.set_ylabel('%')
ax.set_title('Сводное сравнение метрик по группам')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

for bars in [bars_a, bars_b]:
    ax.bar_label(bars, fmt='%.1f%%', padding=3)

plt.tight_layout()
plt.show()

In [ ]:
lift_labels = ['Completion rate', 'Retention D7', 'Retention D14']
lift_values = [
    (b['completion_rate'].mean()/a['completion_rate'].mean() - 1) * 100,
    (b['returned_d7'].mean()/a['returned_d7'].mean() - 1) * 100,
    (b['returned_d14'].mean()/a['returned_d14'].mean() - 1) * 100
]

plt.figure(figsize=(7,4))
bars = plt.barh(lift_labels, lift_values, color='#1D9E75')
plt.xlabel('Относительный прирост, %')
plt.title('Насколько выросла каждая метрика в группе B относительно A')
plt.bar_label(bars, fmt='+%.1f%%', padding=3)
plt.tight_layout()
plt.show()

## Шаг 8. Бизнес-рекомендация

**Вердикт: раскатить геймификацию на всех пользователей платформы.**

За 3 недели эксперимента на очищенной выборке 5980 учеников (2992 в группе A, 2988 в группе B) все три ключевые метрики выросли в группе с геймификацией, эффект статистически значим (p < 0.001 по каждой метрике) и практически ощутим (до +26.6% в относительном выражении).

**Ограничения и следующие шаги:**
- Эксперимент шёл 3 недели, данных о поведении после 14 дня нет — после раскатывания стоит продолжить мониторинг метрик, чтобы отличить устойчивый эффект от временного всплеска интереса.
- Эффект геймификации в данных распределён неравномерно по классам учеников — сегментированный анализ по `grade` не входил в этот отчёт, но может уточнить приоритеты поэтапного запуска.